# Phase 6: Traditional Restoration Methods — All Categories & Degradation Types

## Objective
Apply **four traditional image restoration filters** to all degraded training images
(from Phases 2–5) and evaluate restoration quality against the original clean images
using **MSE**, **PSNR**, and **SSIM**.

### Restoration Methods
| Method | Implementation | Fixed Parameters |
|---|---|---|
| **Median Filter** | `cv2.medianBlur` | kernel = 3×3 |
| **NLM** | `cv2.fastNlMeansDenoising` | h=10, template=7×7, search=21×21 |
| **Wiener Filter** | `scipy.signal.wiener` | window = 5×5 |
| **Lanczos + Sharpening** | `skimage.filters.unsharp_mask` | radius=1.0, amount=1.5 |

### Scope
- **Input:** All degraded training images from Phases 2–5
  - Gaussian (39,200) + Rician (39,200) + Low-Res (39,200) + Motion Blur (39,200) = **156,800** images
- **Reference:** Corresponding clean original training images
- **Output:** Per-image metrics (MSE, PSNR, SSIM) saved to CSV
- **Disk:** Only the **first sample image** per (method × degradation × category × severity) 
  combination is saved for visualization (448 images total).

### Strict Experimental Constraints
- Restoration is applied to degraded images only — **clean originals are untouched**
- Parameters are **fixed** across all severity levels for unbiased comparison
- All processing is **reproducible** (deterministic filters, fixed seeds from prior phases)


In [1]:
import os
import sys
import time
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor
from itertools import chain

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Project root
PROJECT_ROOT = Path(r"C:\Users\gitan\BrainMRI_Project")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.restoration import (
    run_worker_restore,
    RESTORATION_METHODS,
    MEDIAN_KERNEL_SIZE, NLM_H, NLM_TEMPLATE_WINDOW, NLM_SEARCH_WINDOW,
    WIENER_SIZE, LANCZOS_RADIUS, LANCZOS_AMOUNT,
)

print("All imports successful.")
print(f"Restoration methods: {RESTORATION_METHODS}")


All imports successful.
Restoration methods: ['median_filter', 'nlm', 'wiener', 'lanczos_sharpening']


## Restoration Method Parameters

### 1. Median Filter
- **Kernel size:** 3×3
- **Rationale:** Minimum odd kernel that removes isolated noise spikes while preserving anatomical edges.

### 2. Non-Local Means (NLM)
- **h = 10:** Filter strength in uint8 scale (0–255). Standard default for moderate noise.
- **templateWindowSize = 7:** Size of patch used for similarity comparison (7×7 pixels).
- **searchWindowSize = 21:** Extent of the candidate search region (21×21 pixels).
- **Rationale:** Standard cv2 defaults; well-validated for MRI denoising in literature.

### 3. Wiener Filter
- **mysize = 5:** Local estimation window (5×5 pixels).
- **Rationale:** Adaptive MMSE filter; 5×5 window balances noise estimation accuracy and spatial resolution.

### 4. Lanczos + Sharpening (Unsharp Mask)
- **radius = 1.0:** Gaussian blur radius (pixels) for high-frequency residual extraction.
- **amount = 1.5:** Gain applied to the residual. Values >1 over-sharpen, <1 under-sharpen.
- **Rationale:** Recovers detail lost to motion blur and low-resolution downsampling.
  Parameters approximate a Lanczos-type frequency response.


In [2]:
# ============================================================
# Paths
# ============================================================
TRAIN_BASE_DIR   = PROJECT_ROOT / "dataset" / "archive" / "Training"
DEGRADED_BASE    = PROJECT_ROOT / "dataset" / "degraded"

GAUSSIAN_DIR     = DEGRADED_BASE / "gaussian"
RICIAN_DIR       = DEGRADED_BASE / "rician"
LOWRES_DIR       = DEGRADED_BASE / "low_resolution"
MOTION_DIR       = DEGRADED_BASE / "motion_blur"

METRICS_DIR      = PROJECT_ROOT / "outputs" / "metrics"
SAMPLE_DIR       = PROJECT_ROOT / "outputs" / "samples" / "restored"
FIGURES_DIR      = PROJECT_ROOT / "outputs"

METRICS_DIR.mkdir(parents=True, exist_ok=True)
SAMPLE_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================
# Experiment Constants
# ============================================================
CATEGORIES     = ["glioma", "meningioma", "notumor", "pituitary"]
SEVERITY_LEVELS = [10, 20, 30, 40, 50, 60, 70]
NUM_WORKERS    = 12
BASE_SEED      = 42

CLEAN_EXTENSIONS = {".jpg", ".jpeg", ".png"}

# Degradation type → (base_dir, filename suffix template)
DEGRADATION_CONFIG = {
    "gaussian":      (GAUSSIAN_DIR,  "{stem}_gaussian_{sev}.png"),
    "rician":        (RICIAN_DIR,    "{stem}_rician_{sev}.png"),
    "low_resolution":(LOWRES_DIR,    "{stem}_lowres_{sev}.png"),
    "motion_blur":   (MOTION_DIR,    "{stem}_motion_{sev}.png"),
}

print("Paths configured.")
print(f"Categories:      {CATEGORIES}")
print(f"Severity levels: {SEVERITY_LEVELS}")
print(f"Workers:         {NUM_WORKERS}")
print(f"Degradation types: {list(DEGRADATION_CONFIG.keys())}")


Paths configured.
Categories:      ['glioma', 'meningioma', 'notumor', 'pituitary']
Severity levels: [10, 20, 30, 40, 50, 60, 70]
Workers:         12
Degradation types: ['gaussian', 'rician', 'low_resolution', 'motion_blur']


## Pilot Test — Verify Restoration on 1 Image

In [3]:
from src.preprocessing import normalize_image
from src.restoration import apply_all_methods
from src.metrics import calculate_mse, calculate_psnr, calculate_ssim

# Pick first glioma image as pilot
clean_imgs = sorted([p for p in (TRAIN_BASE_DIR / "glioma").iterdir()
                     if p.suffix.lower() in CLEAN_EXTENSIONS])
pilot_clean_path = clean_imgs[0]
pilot_deg_path   = GAUSSIAN_DIR / "glioma" / "50" / f"{pilot_clean_path.stem}_gaussian_50.png"

with Image.open(pilot_clean_path) as img:
    clean_arr = normalize_image(img, target_size=(512, 512))
with Image.open(pilot_deg_path) as img:
    deg_arr = normalize_image(img, target_size=(512, 512))

restored = apply_all_methods(deg_arr)

print(f"Pilot image: {pilot_clean_path.name} @ Gaussian 50%")
print(f"{'Method':<22} {'MSE':>10} {'PSNR (dB)':>12} {'SSIM':>8}")
print("-" * 56)
for method, arr in restored.items():
    m = calculate_mse(clean_arr, arr)
    p = calculate_psnr(clean_arr, arr)
    s = calculate_ssim(clean_arr, arr)
    print(f"{method:<22} {m:10.6f} {p:12.4f} {s:8.4f}")

print("\nPilot test PASSED — all restoration methods functional.")


Pilot image: Tr-gl_1.jpg @ Gaussian 50%
Method                        MSE    PSNR (dB)     SSIM
--------------------------------------------------------
median_filter            0.027407      15.6213   0.0919
nlm                      0.117385       9.3039   0.0254
wiener                   0.029188      15.3479   0.1426
lanczos_sharpening       0.205675       6.8682   0.0140

Pilot test PASSED — all restoration methods functional.


## Main Processing — Build Work Items and Run

In [4]:
def build_work_items(degradation: str, deg_base_dir: Path, suffix_template: str):
    # Build list of (degraded_path, clean_path, ...) tuples for all categories/severities.
    items = []
    for category in CATEGORIES:
        clean_dir   = TRAIN_BASE_DIR / category
        clean_paths = sorted([p for p in clean_dir.iterdir()
                              if p.suffix.lower() in CLEAN_EXTENSIONS])
        for severity in SEVERITY_LEVELS:
            deg_dir = deg_base_dir / category / str(severity)
            for idx, clean_p in enumerate(clean_paths):
                suffix   = suffix_template.format(stem=clean_p.stem, sev=severity)
                deg_path = deg_dir / suffix
                items.append((
                    str(deg_path),
                    str(clean_p),
                    degradation,
                    category,
                    severity,
                    idx,
                    str(SAMPLE_DIR),
                ))
    return items


### Processing: GAUSSIAN

In [5]:
deg_name   = "gaussian"
deg_dir, suffix_tpl = DEGRADATION_CONFIG[deg_name]
csv_path = METRICS_DIR / f"phase6_{deg_name}_metrics.csv"

if csv_path.exists() and len(pd.read_csv(csv_path)) == 156800:
    print(f"Loading existing verified metrics: {csv_path.name} (156,800 rows)")
    df_gaussian = pd.read_csv(csv_path)
else:
    t_start = time.time()
    print(f"Building work items for {deg_name} ...")
    work_items = build_work_items(deg_name, deg_dir, suffix_tpl)
    print(f"  Total items: {len(work_items)} (each spawns 4 method evaluations)")

    all_results_gaussian = []
    with ProcessPoolExecutor(max_workers=NUM_WORKERS) as executor:
        for result_list in executor.map(run_worker_restore, work_items):
            all_results_gaussian.extend(result_list)

    elapsed = time.time() - t_start
    n_rows  = len(all_results_gaussian)
    print(f"  Done: {n_rows} metric rows in {elapsed/60:.1f} min")

    df_gaussian = pd.DataFrame(all_results_gaussian)
    df_gaussian.to_csv(csv_path, index=False)
    print(f"  Saved: {csv_path.name}")

print(f"Metrics summary for {deg_name}:")
print(df_gaussian.groupby(['category','severity','method'])[['mse','psnr','ssim']].mean().round(4).head(12).to_string())


Loading existing verified metrics: phase6_gaussian_metrics.csv (156,800 rows)
Metrics summary for gaussian:
                                         mse     psnr    ssim
category severity method                                     
glioma   10       lanczos_sharpening  0.0286  15.4501  0.0813
                  median_filter       0.0014  28.6841  0.5142
                  nlm                 0.0054  22.7094  0.2780
                  wiener              0.0017  27.7158  0.4818
         20       lanczos_sharpening  0.0859  10.6658  0.0325
                  median_filter       0.0050  23.0416  0.2703
                  nlm                 0.0259  15.8806  0.0820
                  wiener              0.0062  22.0732  0.2765
         30       lanczos_sharpening  0.1388   8.5782  0.0200
                  median_filter       0.0107  19.7264  0.1679
                  nlm                 0.0529  12.7678  0.0463
                  wiener              0.0130  18.8695  0.1897


### Processing: RICIAN

In [6]:
deg_name   = "rician"
deg_dir, suffix_tpl = DEGRADATION_CONFIG[deg_name]
csv_path = METRICS_DIR / f"phase6_{deg_name}_metrics.csv"

if csv_path.exists() and len(pd.read_csv(csv_path)) == 156800:
    print(f"Loading existing verified metrics: {csv_path.name} (156,800 rows)")
    df_rician = pd.read_csv(csv_path)
else:
    t_start = time.time()
    print(f"Building work items for {deg_name} ...")
    work_items = build_work_items(deg_name, deg_dir, suffix_tpl)
    print(f"  Total items: {len(work_items)} (each spawns 4 method evaluations)")

    all_results_rician = []
    with ProcessPoolExecutor(max_workers=NUM_WORKERS) as executor:
        for result_list in executor.map(run_worker_restore, work_items):
            all_results_rician.extend(result_list)

    elapsed = time.time() - t_start
    n_rows  = len(all_results_rician)
    print(f"  Done: {n_rows} metric rows in {elapsed/60:.1f} min")

    df_rician = pd.DataFrame(all_results_rician)
    df_rician.to_csv(csv_path, index=False)
    print(f"  Saved: {csv_path.name}")

print(f"Metrics summary for {deg_name}:")
print(df_rician.groupby(['category','severity','method'])[['mse','psnr','ssim']].mean().round(4).head(12).to_string())


Loading existing verified metrics: phase6_rician_metrics.csv (156,800 rows)
Metrics summary for rician:


                                         mse     psnr    ssim
category severity method                                     
glioma   10       lanczos_sharpening  0.0374  14.2729  0.0731
                  median_filter       0.0081  20.9779  0.3492
                  nlm                 0.0121  19.1764  0.1947
                  wiener              0.0085  20.7626  0.3905
         20       lanczos_sharpening  0.1254   9.0180  0.0262
                  median_filter       0.0360  14.4785  0.1825
                  nlm                 0.0578  12.3965  0.0667
                  wiener              0.0381  14.2439  0.2280
         30       lanczos_sharpening  0.2204   6.5738  0.0137
                  median_filter       0.0879  10.5876  0.1048
                  nlm                 0.1329   8.7784  0.0330
                  wiener              0.0926  10.3694  0.1518


### Processing: LOW RESOLUTION

In [7]:
deg_name   = "low_resolution"
deg_dir, suffix_tpl = DEGRADATION_CONFIG[deg_name]
csv_path = METRICS_DIR / f"phase6_{deg_name}_metrics.csv"

if csv_path.exists() and len(pd.read_csv(csv_path)) == 156800:
    print(f"Loading existing verified metrics: {csv_path.name} (156,800 rows)")
    df_low_resolution = pd.read_csv(csv_path)
else:
    t_start = time.time()
    print(f"Building work items for {deg_name} ...")
    work_items = build_work_items(deg_name, deg_dir, suffix_tpl)
    print(f"  Total items: {len(work_items)} (each spawns 4 method evaluations)")

    all_results_low_resolution = []
    with ProcessPoolExecutor(max_workers=NUM_WORKERS) as executor:
        for result_list in executor.map(run_worker_restore, work_items):
            all_results_low_resolution.extend(result_list)

    elapsed = time.time() - t_start
    n_rows  = len(all_results_low_resolution)
    print(f"  Done: {n_rows} metric rows in {elapsed/60:.1f} min")

    df_low_resolution = pd.DataFrame(all_results_low_resolution)
    df_low_resolution.to_csv(csv_path, index=False)
    print(f"  Saved: {csv_path.name}")

print(f"Metrics summary for {deg_name}:")
print(df_low_resolution.groupby(['category','severity','method'])[['mse','psnr','ssim']].mean().round(4).head(12).to_string())


Loading existing verified metrics: phase6_low_resolution_metrics.csv (156,800 rows)


Metrics summary for low_resolution:
                                         mse     psnr    ssim
category severity method                                     
glioma   10       lanczos_sharpening  0.0002  37.6586  0.9811
                  median_filter       0.0001  39.9174  0.9844
                  nlm                 0.0002  36.8219  0.9388
                  wiener              0.0002  38.3509  0.9641
         20       lanczos_sharpening  0.0001  38.8803  0.9838
                  median_filter       0.0001  40.3611  0.9841
                  nlm                 0.0002  36.9176  0.9378
                  wiener              0.0002  38.5352  0.9635
         30       lanczos_sharpening  0.0001  39.5078  0.9840
                  median_filter       0.0001  40.0420  0.9826
                  nlm                 0.0002  36.6365  0.9358
                  wiener              0.0002  38.2809  0.9619


### Processing: MOTION BLUR

In [8]:
deg_name   = "motion_blur"
deg_dir, suffix_tpl = DEGRADATION_CONFIG[deg_name]
csv_path = METRICS_DIR / f"phase6_{deg_name}_metrics.csv"

if csv_path.exists() and len(pd.read_csv(csv_path)) == 156800:
    print(f"Loading existing verified metrics: {csv_path.name} (156,800 rows)")
    df_motion_blur = pd.read_csv(csv_path)
else:
    t_start = time.time()
    print(f"Building work items for {deg_name} ...")
    work_items = build_work_items(deg_name, deg_dir, suffix_tpl)
    print(f"  Total items: {len(work_items)} (each spawns 4 method evaluations)")

    all_results_motion_blur = []
    with ProcessPoolExecutor(max_workers=NUM_WORKERS) as executor:
        for result_list in executor.map(run_worker_restore, work_items):
            all_results_motion_blur.extend(result_list)

    elapsed = time.time() - t_start
    n_rows  = len(all_results_motion_blur)
    print(f"  Done: {n_rows} metric rows in {elapsed/60:.1f} min")

    df_motion_blur = pd.DataFrame(all_results_motion_blur)
    df_motion_blur.to_csv(csv_path, index=False)
    print(f"  Saved: {csv_path.name}")

print(f"Metrics summary for {deg_name}:")
print(df_motion_blur.groupby(['category','severity','method'])[['mse','psnr','ssim']].mean().round(4).head(12).to_string())


Loading existing verified metrics: phase6_motion_blur_metrics.csv (156,800 rows)
Metrics summary for motion_blur:
                                         mse     psnr    ssim
category severity method                                     
glioma   10       lanczos_sharpening  0.0001  39.7209  0.9805
                  median_filter       0.0002  36.9441  0.9678
                  nlm                 0.0004  34.5908  0.9211
                  wiener              0.0003  35.7523  0.9480
         20       lanczos_sharpening  0.0006  32.4403  0.9201
                  median_filter       0.0007  31.9955  0.9161
                  nlm                 0.0008  31.3118  0.8870
                  wiener              0.0007  31.6828  0.9070
         30       lanczos_sharpening  0.0012  29.6942  0.8757
                  median_filter       0.0012  29.5435  0.8748
                  nlm                 0.0013  29.2711  0.8568
                  wiener              0.0012  29.4230  0.8702


## Compile Master Metrics CSV

In [9]:
# Load all intermediate CSVs in case any cells were re-run independently
master_frames = []
for deg_name in DEGRADATION_CONFIG.keys():
    csv_path = METRICS_DIR / f"phase6_{deg_name}_metrics.csv"
    if csv_path.exists():
        master_frames.append(pd.read_csv(csv_path))
        print(f"Loaded {csv_path.name}: {len(master_frames[-1])} rows")

df_master = pd.concat(master_frames, ignore_index=True)
master_csv = METRICS_DIR / "phase6_restoration_metrics_master.csv"
df_master.to_csv(master_csv, index=False)
print(f"\nMaster CSV: {master_csv.name}")
print(f"Total rows: {len(df_master)}")
print(f"Expected:   {4 * 4 * 7 * 1400 * 4} (4 deg × 4 cat × 7 sev × 1400 images × 4 methods)")
print(f"\nColumn summary:")
print(df_master.dtypes)


Loaded phase6_gaussian_metrics.csv: 156800 rows
Loaded phase6_rician_metrics.csv: 156800 rows


Loaded phase6_low_resolution_metrics.csv: 156800 rows
Loaded phase6_motion_blur_metrics.csv: 156800 rows



Master CSV: phase6_restoration_metrics_master.csv
Total rows: 627200
Expected:   627200 (4 deg × 4 cat × 7 sev × 1400 images × 4 methods)

Column summary:
degradation        str
category           str
severity         int64
method             str
filename           str
mse            float64
psnr           float64
ssim           float64
dtype: object


## Summary Statistics

In [10]:
# Aggregate: mean ± std per (degradation, category, severity, method)
df_summary = (
    df_master.groupby(['degradation','category','severity','method'])
    .agg(
        mean_mse  =('mse',  'mean'),
        std_mse   =('mse',  'std'),
        mean_psnr =('psnr', 'mean'),
        std_psnr  =('psnr', 'std'),
        mean_ssim =('ssim', 'mean'),
        std_ssim  =('ssim', 'std'),
        n_images  =('mse',  'count'),
    )
    .reset_index()
)

summary_csv = METRICS_DIR / "phase6_restoration_summary.csv"
df_summary.to_csv(summary_csv, index=False)
print(f"Summary CSV: {summary_csv.name}")
print(f"Rows:        {len(df_summary)}  (expected: {4*4*7*4} = 448)")
print()
print("Best PSNR per degradation type (averaged over all categories and severities):")
best = (df_summary.groupby(['degradation','method'])
        ['mean_psnr'].mean().round(2)
        .reset_index().sort_values(['degradation','mean_psnr'], ascending=[True, False]))
print(best.to_string(index=False))


Summary CSV: phase6_restoration_summary.csv
Rows:        448  (expected: 448 = 448)

Best PSNR per degradation type (averaged over all categories and severities):
   degradation             method  mean_psnr
      gaussian             wiener      18.64
      gaussian      median_filter      18.55
      gaussian                nlm      12.32
      gaussian lanczos_sharpening       8.73
low_resolution lanczos_sharpening      37.35
low_resolution      median_filter      36.64
low_resolution             wiener      34.94
low_resolution                nlm      34.18
   motion_blur lanczos_sharpening      27.97
   motion_blur      median_filter      27.44
   motion_blur             wiener      27.12
   motion_blur                nlm      26.95
        rician             wiener      10.37
        rician      median_filter      10.11
        rician                nlm       8.81
        rician lanczos_sharpening       6.91


## Visualization — PSNR and SSIM vs Severity

In [11]:
METHOD_LABELS = {
    'median_filter':      'Median Filter',
    'nlm':                'NLM',
    'wiener':             'Wiener Filter',
    'lanczos_sharpening': 'Lanczos+Sharp',
}
METHOD_COLORS = {
    'median_filter':      '#1f77b4',
    'nlm':                '#ff7f0e',
    'wiener':             '#2ca02c',
    'lanczos_sharpening': '#d62728',
}
DEGRADATION_LABELS = {
    'gaussian':       'Gaussian Noise',
    'rician':         'Rician Noise',
    'low_resolution': 'Low Resolution',
    'motion_blur':    'Motion Blur',
}

fig, axes = plt.subplots(4, 2, figsize=(16, 22))
fig.suptitle(
    "Phase 6 — Traditional Restoration: PSNR & SSIM vs Severity\n"
    "(averaged over all 4 categories, all 1,400 images per severity)",
    fontsize=14, fontweight='bold', y=1.01
)

for row_idx, deg in enumerate(["gaussian", "rician", "low_resolution", "motion_blur"]):
    df_deg = df_summary[df_summary['degradation'] == deg]
    df_cat_avg = (df_deg.groupby(['severity','method'])
                  [['mean_psnr','mean_ssim']].mean().reset_index())

    for col_idx, metric in enumerate(['mean_psnr', 'mean_ssim']):
        ax = axes[row_idx, col_idx]
        for method in RESTORATION_METHODS:
            sub = df_cat_avg[df_cat_avg['method'] == method].sort_values('severity')
            ax.plot(sub['severity'], sub[metric],
                    marker='o', linewidth=2, markersize=6,
                    label=METHOD_LABELS[method],
                    color=METHOD_COLORS[method])
        ax.set_title(f"{DEGRADATION_LABELS[deg]} — {'PSNR (dB)' if metric=='mean_psnr' else 'SSIM'}",
                     fontsize=11)
        ax.set_xlabel("Severity (%)")
        ax.set_ylabel("PSNR (dB)" if metric == 'mean_psnr' else "SSIM")
        ax.set_xticks(SEVERITY_LEVELS)
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=9)

plt.tight_layout()
fig_path = FIGURES_DIR / "phase6_restoration_psnr_ssim_all.png"
fig.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Figure saved: {fig_path}")


Figure saved: C:\Users\gitan\BrainMRI_Project\outputs\phase6_restoration_psnr_ssim_all.png


C:\Users\gitan\AppData\Local\Temp\ipykernel_12888\1432387896.py:51: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [12]:
# Visual comparison: Clean | Degraded | Median | NLM | Wiener | Lanczos
# Show for one image per degradation type at severity 50%

from src.preprocessing import normalize_image
from src.restoration import apply_all_methods

SAMPLE_CATEGORY = "glioma"
SAMPLE_SEVERITY = 50
clean_dir     = TRAIN_BASE_DIR / SAMPLE_CATEGORY
pilot_clean   = sorted([p for p in clean_dir.iterdir() if p.suffix.lower() in CLEAN_EXTENSIONS])[0]

with Image.open(pilot_clean) as img:
    clean_arr = normalize_image(img, target_size=(512, 512))

fig, axes = plt.subplots(4, 6, figsize=(22, 16))
fig.suptitle(f"Sample Restorations — Category: {SAMPLE_CATEGORY.title()} | Severity: {SAMPLE_SEVERITY}%",
             fontsize=13, fontweight='bold')

for row_idx, (deg, (deg_dir, suffix_tpl)) in enumerate(DEGRADATION_CONFIG.items()):
    suffix   = suffix_tpl.format(stem=pilot_clean.stem, sev=SAMPLE_SEVERITY)
    deg_path = deg_dir / SAMPLE_CATEGORY / str(SAMPLE_SEVERITY) / suffix
    with Image.open(deg_path) as img:
        deg_arr = normalize_image(img, target_size=(512, 512))
    restored = apply_all_methods(deg_arr)

    images  = [clean_arr, deg_arr] + [restored[m] for m in RESTORATION_METHODS]
    titles  = ['Clean', f'{DEGRADATION_LABELS[deg]}\n({SAMPLE_SEVERITY}%)'] + \
              [METHOD_LABELS[m] for m in RESTORATION_METHODS]

    for col_idx, (arr, title) in enumerate(zip(images, titles)):
        ax = axes[row_idx, col_idx]
        ax.imshow(arr, cmap='gray', vmin=0, vmax=1)
        ax.set_title(title, fontsize=9)
        ax.axis('off')

plt.tight_layout()
fig2_path = FIGURES_DIR / "phase6_sample_restorations.png"
fig.savefig(fig2_path, dpi=120, bbox_inches='tight')
plt.show()
print(f"Figure saved: {fig2_path}")


C:\Users\gitan\anaconda3\envs\brainmri\Lib\site-packages\scipy\signal\_signaltools.py:1728: RuntimeWarning: divide by zero encountered in divide
  res *= (1 - noise / lVar)
C:\Users\gitan\anaconda3\envs\brainmri\Lib\site-packages\scipy\signal\_signaltools.py:1728: RuntimeWarning: invalid value encountered in multiply
  res *= (1 - noise / lVar)


Figure saved: C:\Users\gitan\BrainMRI_Project\outputs\phase6_sample_restorations.png


C:\Users\gitan\AppData\Local\Temp\ipykernel_12888\2133834702.py:39: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Final Verification

In [13]:
print("=" * 60)
print("PHASE 6 — TRADITIONAL RESTORATION: FINAL VERIFICATION")
print("=" * 60)
print()

# Check CSV files
for name in ["phase6_restoration_metrics_master.csv", "phase6_restoration_summary.csv"]:
    path = METRICS_DIR / name
    if path.exists():
        df_chk = pd.read_csv(path)
        print(f"[OK] {name}")
        print(f"     Rows: {len(df_chk)}")
    else:
        print(f"[MISSING] {name}")

print()
print("Summary (mean PSNR per degradation × method, averaged over categories & severities):")
pivot = (df_summary.groupby(['degradation','method'])['mean_psnr']
         .mean().round(2).unstack('method'))
print(pivot.to_string())
print()

# Count sample images
sample_count = len(list(SAMPLE_DIR.rglob("*.png")))
print(f"Sample restored images saved: {sample_count}")
print(f"Expected sample count:        {len(RESTORATION_METHODS) * 4 * 4 * 7}")
print()
print("Phase 6 COMPLETE. Ready for Phase 7 (Metrics & Training Comparison).")


PHASE 6 — TRADITIONAL RESTORATION: FINAL VERIFICATION



[OK] phase6_restoration_metrics_master.csv
     Rows: 627200
[OK] phase6_restoration_summary.csv
     Rows: 448

Summary (mean PSNR per degradation × method, averaged over categories & severities):
method          lanczos_sharpening  median_filter    nlm  wiener
degradation                                                     
gaussian                      8.73          18.55  12.32   18.64
low_resolution               37.35          36.64  34.18   34.94
motion_blur                  27.97          27.44  26.95   27.12
rician                        6.91          10.11   8.81   10.37

Sample restored images saved: 448
Expected sample count:        448

Phase 6 COMPLETE. Ready for Phase 7 (Metrics & Training Comparison).
